# P4 Agent & Tools — Final Colab Setup

This notebook runs the P4 workstream and prepares the received P3 company archives. It does **not** replace P3's retrieval/index implementation.

## 1. Upload and extract the P4 package ZIP

In [ ]:
from google.colab import files
import zipfile, os
uploaded = files.upload()  # Select Competitive_Research_Copilot_P4_Final.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content')
project_dir = '/content/Competitive_Research_Copilot_P4_Final'
os.chdir(project_dir)
print('Project:', os.getcwd())
print('requirements.txt:', os.path.isfile('requirements.txt'))


## 2. Install P4 dependencies

In [ ]:
!pip -q install -r requirements.txt

## 3. Run P4 unit tests

In [ ]:
!PYTHONPATH=. pytest -q

## 4. Inspect the ten-company P3 corpus manifest

In [ ]:
import pandas as pd
manifest = pd.read_csv('data/p3_manifest/p3_corpus_manifest.csv')
display(manifest.groupby(['company_archive','document_type_detected']).size().reset_index(name='pdf_count'))
print('Total annual-report PDFs:', len(manifest))
print('Filename years marked as candidates for FY20-FY24:', (manifest['candidate_in_plan_FY20_FY24']=='yes').sum())


## 5. Optional: stage P3 company report ZIPs directly from your computer

This step does not use Google Drive. Run the code cell below and select the company ZIP archives from your computer. For a large collection, upload 2–3 ZIPs per run, then run the same cell again for the remaining archives. Extracted files remain in the Colab runtime between runs, but are cleared when the runtime is reset.

Expected archive names: `Britannia.zip`, `Colgate.zip`, `Dabur.zip`, `Emami.zip`, `Godrej Consumer.zip`, `HUL.zip`, `ITC.zip`, `Marico.zip`, `Nestle.zip`, and `Tata Consumer.zip`. You can upload the archives available now and add the rest later.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

corpus_root = Path('/content/p3_company_corpus')
corpus_root.mkdir(parents=True, exist_ok=True)

# Select 2–3 company ZIP files at a time. Run this cell again for the next batch.
uploaded = files.upload()

for name, data in uploaded.items():
    if not name.lower().endswith('.zip'):
        print('Skipped (not a ZIP):', name)
        continue

    archive_path = Path('/content') / Path(name).name
    archive_path.write_bytes(data)
    destination = corpus_root / Path(name).stem
    destination.mkdir(parents=True, exist_ok=True)

    try:
        with zipfile.ZipFile(archive_path, 'r') as archive:
            bad_member = archive.testzip()
            if bad_member is not None:
                raise zipfile.BadZipFile(f'Corrupt member: {bad_member}')
            # Reject paths that would extract outside the company folder.
            root = destination.resolve()
            for member in archive.infolist():
                target = (destination / member.filename).resolve()
                if target != root and root not in target.parents:
                    raise ValueError(f'Unsafe ZIP path: {member.filename}')
            archive.extractall(destination)
        print('Extracted:', name, '->', destination)
    except Exception as exc:
        print('Could not extract', name, ':', type(exc).__name__, str(exc))
    finally:
        archive_path.unlink(missing_ok=True)

pdfs = list(corpus_root.rglob('*.pdf'))
print('PDFs currently staged:', len(pdfs))
print('Company folders:', sum(1 for p in corpus_root.iterdir() if p.is_dir()))
print('Corpus staging folder:', corpus_root)

## 6. Handoff check — P3 retrieval code is still required

The uploaded ten ZIPs contain company annual-report PDFs. They do not contain the P3 parser, chunker, BM25/dense/FAISS/RRF/reranker, table extraction or retrieval API. Do not run a live P4 search until P3 supplies those modules and confirms the callable signatures.

In [ ]:
from pathlib import Path

corpus_root = Path('/content/p3_company_corpus')
if corpus_root.exists():
    print('P3 staged PDFs:', len(list(corpus_root.rglob('*.pdf'))))
    for company_dir in sorted(p for p in corpus_root.iterdir() if p.is_dir()):
        print(f'{company_dir.name}: {len(list(company_dir.rglob("*.pdf")))} PDFs')
else:
    print('P3 staged PDFs: 0 (upload the company ZIP files in section 5 when ready)')

print('P3 retrieval adapter module:', Path('p4/tools/retrieval.py').exists())
print('P3 retrieval is connected only after retrieve_fn and table_fn are injected from P3.')

## 7. Next integration handoff

In [ ]:
from pathlib import Path
print(Path('integration/OWNER_HANDOFF.md').read_text())
